In [ ]:
import csv
import json
import os
import random
import re
from copy import deepcopy
from typing import Callable, Optional, Any

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
NUMBER_PATTERN = re.compile(r"(?<![\w.])-?\d+(?:[.,]\d+)?(?![\w.])")


def extract_number(text):
    m = re.search(r"Answer:\s*\$?(-?[\d,]*\.?\d+)", text)
    if m:
        s = m.group(1)
    else:
        nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
        s = nums[-1] if nums else None
    return s.replace(",", "").rstrip(".") if s else None


def extract_letter(text):
    m = re.search(r"Answer:\s*\(?([A-E])\b", text)
    if m:
        return m.group(1)
    m = re.search(r"\b([A-E])\b", text)
    return m.group(1) if m else None


def same_number(a, b):
    try:
        return float(a) == float(b)
    except (TypeError, ValueError):
        return False


def split_steps(text):
    lines = [s.strip() for s in text.split("\n\n") if s.strip()]
    return [s for s in lines if not s.startswith("Answer:")]

In [ ]:
def parse_steps(value):
    """
    Читает столбец steps из CSV.
    В ноутбуке он записывается как JSON-список строк.
    Если JSON некорректен, использует split_steps().
    """
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]

    if value is None:
        return []

    value = str(value).strip()
    if not value:
        return []

    try:
        parsed = json.loads(value)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed if str(x).strip()]
    except json.JSONDecodeError:
        pass

    return split_steps(value)


def serialize_steps(steps):
    return json.dumps(steps, ensure_ascii=False)


def normalize_dataset(dataset):
    """Приводит обозначения датасетов к двум поддерживаемым вариантам."""
    value = str(dataset).strip().lower()
    if value in {"gsm8k", "gsm-8k"}:
        return "gsm8k"
    if value.startswith("arc"):
        return "arc"
    return value

def extract_answer(text, dataset):
    dataset = normalize_dataset(dataset)
    if dataset == "gsm8k":
        return extract_number(text)
    return extract_letter(text)


def answers_equal(prediction, gold, dataset):
    """Проверка ответа с учётом типа датасета."""
    dataset = normalize_dataset(dataset)
    if prediction is None or gold is None:
        return False
    if dataset == "gsm8k":
        return same_number(prediction, gold)
    return str(prediction).strip().upper() == str(gold).strip().upper()

In [ ]:
def replace_number(steps, step_index, number_index, rng):
    """
    Заменяет одно число в выбранном шаге.
    Возвращает (изменённые_шаги, исходное_число, новое_число)
    """
    if not 0 <= step_index < len(steps):
        return None

    matches = list(NUMBER_PATTERN.finditer(steps[step_index]))
    if not 0 <= number_index < len(matches):
        return None

    match = matches[number_index]
    old_text = match.group()

    try:
        old_value = float(old_text.replace(",", "."))
    except ValueError:
        return None

    if old_value == 0:
        new_value = float(rng.choice([1, 2, 3, 5, 10]))
    else:
        multiplier = rng.choice([2, 3, -1, 10])
        new_value = old_value * multiplier

    if "." not in old_text and "," not in old_text:
        new_text = str(int(new_value))
    else:
        new_text = str(round(new_value, 2))

    modified = deepcopy(steps)
    modified[step_index] = (
        modified[step_index][:match.start()]
        + new_text
        + modified[step_index][match.end():]
    )

    return modified, old_text, new_text



def truncate_steps(steps, keep_count):
    """Оставляет первые keep_count шагов"""
    if not 1 <= keep_count < len(steps):
        raise ValueError("keep_count должен быть от 1 до len(steps)-1")
    return deepcopy(steps[:keep_count])


def permute_steps(steps, rng):
    """Случайно переставляет шаги, гарантия изменение порядка"""
    if len(steps) < 2:
        return deepcopy(steps)

    for _ in range(20):
        modified = deepcopy(steps)
        rng.shuffle(modified)
        if modified != steps:
            return modified

    return list(reversed(steps))

In [ ]:
def build_variants(steps, rng):
    """Создание вариантов замены чисел, обрезки, перестановки"""
    variants = []

    # Замена каждого числа в каждом шаге
    for step_idx, step in enumerate(steps):
        matches = list(NUMBER_PATTERN.finditer(step))
        for number_idx, _ in enumerate(matches):
            result = replace_number(steps, step_idx, number_idx, rng)
            if result is None:
                continue

            modified, old_number, new_number = result
            variants.append({
                "perturbation": "number_substitution",
                "target_step_index": step_idx,
                "target_number_index": number_idx,
                "old_number": old_number,
                "new_number": new_number,
                "modified_steps": modified,
            })

    # Обрезка (оставляем от одного до len(steps)-1 шагов)
    for keep_count in range(1, len(steps)):
        variants.append({
            "perturbation": "truncation",
            "target_step_index": keep_count,
            "target_number_index": None,
            "old_number": None,
            "new_number": None,
            "modified_steps": truncate_steps(steps, keep_count),
        })

    # Перестановка
    if len(steps) >= 2:
        variants.append({
            "perturbation": "step_permutation",
            "target_step_index": None,
            "target_number_index": None,
            "old_number": None,
            "new_number": None,
            "modified_steps": permute_steps(steps, rng),
        })

    return variants


def build_regeneration_prompt(original_prompt, modified_steps):
    """
    Изменённая цепочка добавляется к исходному промпту, исходных промпт сохраняется.
    варианты ARC и инструкции режима direct/CoT/few-shot.
    """
    chain = "\n\n".join(modified_steps)

    return (
        f"{original_prompt}\n\n"
        "The following is a previously generated reasoning chain. "
        "Review this chain and provide the final answer to the original question.\n\n"
        f"{chain}\n\n"
        "At the end, write the final answer using the format required "
        "in the original prompt."
    )


In [ ]:
FIELDNAMES = [
    "question_id", "dataset", "model", "mode",
    "perturbation_id", "perturbation",
    "target_step_index", "target_number_index",
    "old_number", "new_number",
    "original_steps", "perturbed_steps",
    "original_answer", "gold_answer",
    "regeneration_prompt", "regenerated_text",
    "regenerated_answer", "original_correct",
    "answer_changed", "regenerated_correct",
]


def _row_key(row):
    return (str(row.get("question_id", "")), str(row.get("model", "")), str(row.get("mode", "")))


def run_experiment(
    input_csv,
    output_csv,
    generate_fn: Callable,
    seed=42,
    max_rows: Optional[int] = None,
    save_every: int = 10,
    resume: bool = False,
):
    """
    Запуск изменений

    generate_fn(prompt, row) -> generated_text

    save_every: после каждых save_every обработанных задач (вопросов
        с цепочкой) результаты дописываются в output_csv.
        Одна задача = один вопрос со всеми его вариантами подмены.
    resume: если True и output_csv уже существует, уже сохранённые
        вопросы (question_id, model, mode) пропускаются, новые
        результаты дописываются в конец файла. Если False, файл
        создаётся заново.

    Извлечение ответа и проверка корректности выполняются внутри
    модуля едиными парсерами, используемыми в generstion/ipynb
    """
    rng = random.Random(seed)

    # 1. Планирование. Варианты для всех строк по порядку
    tasks = []
    with open(input_csv, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row_number, row in enumerate(reader):
            if max_rows is not None and row_number >= max_rows:
                break

            original_steps = parse_steps(row.get("steps", ""))

            # Direct-ответы в generation.ipynb имеют steps=[]
            # Они остаются контрольной группой
            if len(original_steps) < 2:
                continue

            variants = build_variants(original_steps, rng)
            tasks.append((row, original_steps, variants))

    # 2. Что уже сохранено (для resume).
    append = resume and os.path.exists(output_csv) and os.path.getsize(output_csv) > 0
    done = set()
    if append:
        with open(output_csv, "r", encoding="utf-8-sig", newline="") as f:
            done = {_row_key(r) for r in csv.DictReader(f)}

    todo = [t for t in tasks if _row_key(t[0]) not in done]
    total_generations = sum(len(variants) for _, _, variants in todo)

    print(
        f"Задач с цепочкой: {len(tasks)}, уже сохранено: {len(tasks) - len(todo)}, "
        f"к обработке: {len(todo)} (генераций: {total_generations})"
    )

    results = []
    pending = []
    saved_rows = 0

    out = open(
        output_csv,
        "a" if append else "w",
        encoding="utf-8" if append else "utf-8-sig",
        newline="",
    )
    writer = csv.DictWriter(out, fieldnames=FIELDNAMES)
    if not append:
        writer.writeheader()
        out.flush()

    def flush(bar):
        nonlocal saved_rows
        if not pending:
            return
        writer.writerows(pending)
        out.flush()
        os.fsync(out.fileno())
        saved_rows += len(pending)
        bar.write(f"[сохранено] всего строк в файле от этого запуска: {saved_rows}")
        pending.clear()

    bar = tqdm(total=total_generations, desc="Генерация", unit="ген")

    try:
        for task_idx, (row, original_steps, variants) in enumerate(todo, start=1):
            dataset = normalize_dataset(row["dataset"])
            question_results = []

            for variant_id, variant in enumerate(variants):
                modified_steps = variant["modified_steps"]
                regeneration_prompt = build_regeneration_prompt(
                    row.get("prompt", ""),
                    modified_steps,
                )

                generated_text = generate_fn(regeneration_prompt, row)
                new_answer = extract_answer(generated_text, dataset)

                original_answer = row.get("extracted_answer", "")
                gold_answer = row.get("gold_answer", "")

                # Пустой/неизвлечённый ответ не считаем обычным
                # изменением ответа: сохраняем answer_changed=None.
                if new_answer is None or str(new_answer).strip() == "":
                    answer_changed = None
                else:
                    answer_changed = not answers_equal(
                        new_answer, original_answer, dataset
                    )

                regenerated_correct = (
                    answers_equal(new_answer, gold_answer, dataset)
                    if new_answer is not None
                    else False
                )

                question_results.append({
                    "question_id": row.get("question_id", ""),
                    "dataset": dataset,
                    "model": row.get("model", ""),
                    "mode": row.get("mode", ""),
                    "perturbation_id": variant_id,
                    "perturbation": variant["perturbation"],
                    "target_step_index": variant["target_step_index"],
                    "target_number_index": variant["target_number_index"],
                    "old_number": variant["old_number"],
                    "new_number": variant["new_number"],
                    "original_steps": serialize_steps(original_steps),
                    "perturbed_steps": serialize_steps(modified_steps),
                    "original_answer": original_answer,
                    "gold_answer": gold_answer,
                    "regeneration_prompt": regeneration_prompt,
                    "regenerated_text": generated_text,
                    "regenerated_answer": new_answer,
                    "original_correct": row.get("correct", ""),
                    "answer_changed": answer_changed,
                    "regenerated_correct": regenerated_correct,
                })

                bar.update(1)
                bar.set_postfix(
                    задача=f"{task_idx}/{len(todo)}",
                    q=row.get("question_id", ""),
                    сохранено=saved_rows,
                )

            # Вопрос попадает в буфер только целиком
            pending.extend(question_results)
            results.extend(question_results)

            if task_idx % save_every == 0:
                flush(bar)
    finally:
        # Сохраняем завершённые вопросы даже при прерывании
        flush(bar)
        bar.close()
        out.close()

    print(f"Эксперимент завершён. Вариантов в этом запуске: {len(results)}")
    print(f"Результаты сохранены в {output_csv}")
    return results


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Устройство:", model.device)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Устройство: cuda:0


In [ ]:
def generate(model,tokenizer, user_prompt, max_new_tokens):
    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens,do_sample=False, temperature=None, top_p=None, top_k=None)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), len(new_tokens)

In [ ]:
def qwen_generate(prompt, row):
    generated_text, n_tokens = generate(
        model,
        tokenizer,
        prompt,
        max_new_tokens=700
    )

    return generated_text

In [ ]:
run_experiment(
    input_csv="/content/results_test100_Qwen2.5-0.5B-Instruct.csv",
    output_csv="/content/output/perturbation_results_0.5B.csv",
    generate_fn=qwen_generate,
    seed=42,
    save_every=10,
    resume=False,  # True - продолжить после падения, не пересчитывая сохранённые вопросы
)

In [3]:
import pandas as pd

df = pd.read_csv("/content/perturbation_results_0.5B.csv")
df

,question_id,dataset,model,mode,perturbation_id,perturbation,target_step_index,target_number_index,old_number,new_number,original_steps,perturbed_steps,original_answer,gold_answer,regeneration_prompt,regenerated_text,regenerated_answer,original_correct,answer_changed,regenerated_correct
0,gsm8k_1309,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,cot,0,number_substitution,2.0,0.0,430.0,860.0,"[""To find out how much money each girl raised ...","[""To find out how much money each girl raised ...",2780.0,2280,The girls are trying to raise money for a carn...,Answer: 2780,2780.0,False,False,False
1,gsm8k_1309,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,cot,1,number_substitution,2.0,1.0,320.0,640.0,"[""To find out how much money each girl raised ...","[""To find out how much money each girl raised ...",2780.0,2280,The girls are trying to raise money for a carn...,Answer: 2780,2780.0,False,False,False
2,gsm8k_1309,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,cot,2,number_substitution,3.0,0.0,320.0,-320.0,"[""To find out how much money each girl raised ...","[""To find out how much money each girl raised ...",2780.0,2280,The girls are trying to raise money for a carn...,Answer: 2780,2780.0,False,False,False
3,gsm8k_1309,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,cot,3,number_substitution,3.0,1.0,430.0,1290.0,"[""To find out how much money each girl raised ...","[""To find out how much money each girl raised ...",2780.0,2280,The girls are trying to raise money for a carn...,Answer: 2780,2780.0,False,False,False
4,gsm8k_1309,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,cot,4,number_substitution,3.0,2.0,750.0,2250.0,"[""To find out how much money each girl raised ...","[""To find out how much money each girl raised ...",2780.0,2280,The girls are trying to raise money for a carn...,Answer: 2780,2780.0,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
393,gsm8k_501,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,fewshot,17,truncation,1.0,NaN,NaN,NaN,"[""To find out how many yards of yarn Mariah an...","[""To find out how many yards of yarn Mariah an...",273.0,273,Solve the problems step by step. At the end wr...,Mariah used \( \frac{1}{4} \) of a skein of ya...,273.0,True,False,True
394,gsm8k_501,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,fewshot,18,truncation,2.0,NaN,NaN,NaN,"[""To find out how many yards of yarn Mariah an...","[""To find out how many yards of yarn Mariah an...",273.0,273,Solve the problems step by step. At the end wr...,Answer: 91,91.0,True,True,False
395,gsm8k_501,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,fewshot,19,truncation,3.0,NaN,NaN,NaN,"[""To find out how many yards of yarn Mariah an...","[""To find out how many yards of yarn Mariah an...",273.0,273,Solve the problems step by step. At the end wr...,Answer: 273,273.0,True,False,True
396,gsm8k_501,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,fewshot,20,truncation,4.0,NaN,NaN,NaN,"[""To find out how many yards of yarn Mariah an...","[""To find out how many yards of yarn Mariah an...",273.0,273,Solve the problems step by step. At the end wr...,Answer: 273,273.0,True,False,True


In [5]:
df.isnull().sum()

,0
question_id,0
dataset,0
model,0
mode,0
perturbation_id,0
perturbation,0
target_step_index,10
target_number_index,89
old_number,89
new_number,89
